# 03 — Blockchain Transactions Extraction
## Extraction de TOUTES les transactions Bitcoin depuis genesis

**⚠️ Ce notebook tourne pendant 5-10 JOURS en continu.**
Utiliser `nohup papermill` ou `screen` pour éviter les interruptions.

```bash
nohup papermill notebooks/03_blockchain_transactions.ipynb \
      logs/03_output_$(date +%Y%m%d).ipynb \
      > logs/03_stderr.log 2>&1 &
```

**Volume** : ~900M transactions → ~130 GB Parquet
**Métriques par transaction** : CDD, fee rate, type (consolidation/batch), SegWit/Taproot

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Imports et configuration
# ════════════════════════════════════════════════════════════════════════
import os, sys, time
sys.path.insert(0, "/workspace")

from btc_pipeline.storage.gcs_client import StorageClient
from btc_pipeline.config import Config
from btc_pipeline.collectors.bitcoin_core import (
    get_rpc, get_chain_info, run_transactions_extraction,
)

os.environ.setdefault("GCS_BUCKET_NAME", "btc-training-data")
os.environ.setdefault("STORAGE_BACKEND", "gcs")
os.environ.setdefault("WORKSPACE_DIR",  "/workspace")

storage = StorageClient()
config = Config()

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Vérifications
# ════════════════════════════════════════════════════════════════════════
rpc = get_rpc()
info = get_chain_info(rpc)
state = storage.get_pipeline_state()
last_block = state.get("tasks", {}).get("blockchain_transactions", {}).get("last_completed_block_height", -1)
total_txs = state.get("tasks", {}).get("blockchain_transactions", {}).get("batches_uploaded", 0) * 100000

remaining_blocks = info["blocks"] - last_block
# Estimation : ~4000 tx/s average, blocks have varying tx counts
est_days = remaining_blocks * 3000 / 100000 / (4000 * 86400)

print(f"📊 Transactions extraction")
print(f"   Dernier bloc traité   : {last_block:,}")
print(f"   Blocs restants        : {remaining_blocks:,}")
print(f"   Transactions estimées : ~{remaining_blocks * 3000:,.0f}")
print(f"   Batches déjà uploadés : {total_txs // 100000}")
print(f"   ⏱️  Estimation         : ~{est_days:.0f} jours")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Extraction (LONGUE DURÉE — 5-10 jours)
# ════════════════════════════════════════════════════════════════════════
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(message)s")

print("▶ Lancement de l'extraction des transactions...")
print("  Ctrl+C pour arrêter proprement (l'état sera sauvegardé)")
print()

try:
    stats = run_transactions_extraction(
        storage,
        batch_size=100_000,
        config=config,
    )

    print(f"\n{'═' * 55}")
    print(f"RÉSUMÉ TRANSACTIONS")
    print(f"{'═' * 55}")
    print(f"  Transactions : {stats['total_rows']:,}")
    print(f"  Volume GCS   : {stats['total_gb']:.2f} GB")
    print(f"  Durée        : {stats['duration_s']/86400:.1f} jours")
    print(f"  Batches      : {stats['batches']}")
    print(f"{'═' * 55}")

except KeyboardInterrupt:
    print("\n⏹  Arrêt demandé — état sauvegardé dans pipeline_state.json")
    print("  Relancer ce notebook pour reprendre.")